In [ ]:
import numpy as np
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation

from ev_sim.distance import distance

In [ ]:
gdf = gpd.read_file("data/semcog/semcog_region-Blockgroup-20251120.geojson")

# gdf is a dataframe of geographical zones represented as MultiPolygons
# extract the centroids and lat/lon bounds of each zone
gdf["lat"] = np.radians(gdf["geometry"].apply(lambda x: x.centroid.coords[0][1]))
gdf["lon"] = np.radians(gdf["geometry"].apply(lambda x: x.centroid.coords[0][0]))
gdf["area"] = gdf["geometry"].apply(lambda x: x.area)
gdf["area_frac"] = gdf["area"] / gdf["area"].sum()

gdf.plot(column="daily_trips", cmap="viridis")
gdf.head()

In [ ]:
lat = gdf["lat"].to_numpy()
lon = gdf["lon"].to_numpy()

lat1 = lat[:, None]  # shape (N, 1)
lon1 = lon[:, None]
lat2 = lat[None, :]  # shape (1, N)
lon2 = lon[None, :]

T = distance(lat1, lon1, lat2, lon2, is_radians=False)
Q = T / 40  # kmph
np.fill_diagonal(Q, 2 / 60)  # flat 2 minutes travel time within block

In [ ]:
H = len(gdf)
N = 40000

lmbda = gdf["daily_trips"].values.astype(float) / 24  # per hour


def P(nI):
    mask = (nI >= 1)
    Tm = np.where(mask[:, None], T, np.inf)

    min_per_col = Tm.min(axis=0)
    valid_cols = np.isfinite(min_per_col)
    argmin_per_col = Tm.argmin(axis=0)

    P = np.zeros_like(T, dtype=np.uint8)
    cols = np.nonzero(valid_cols)[0]
    P[argmin_per_col[cols], cols] = 1

    return P


def dnI_dt(nI_k, nT_k, nP_k):
    return (nT_k / T).sum(axis=0) - (lmbda[None, :] * P(nI_k)).sum(axis=1)

def dnT_dt(nI_k, nT_k, nP_k):
    return (nP_k / Q).sum(axis=0)[:, None] / H - nT_k / T

def dnP_dt(nI_k, nT_k, nP_k):
    return lmbda[None, :] * P(nI_k) - nP_k / Q

In [ ]:
nI = N * gdf["area_frac"].values # idle vehicles
nT = np.zeros((H, H))  # first-mile vehicles
nP = np.zeros((H, H))  # last-mile vehicles

h = 0.001
n = 4000

print(f"    Time step: {h * 3600} secs")
print(f"Simulating to: {h * n} hrs")

nI_history = []
nT_history = []
nP_history = []
history_interval = 4

# fourth-order Runge-Kutta scheme
for i in range(n):
    if i % history_interval == 0:
        nI_history.append(nI)
        nT_history.append(nT)
        nP_history.append(nP)

    X = (nI, nT, nP)
    nI_k1 = dnI_dt(*X)
    nT_k1 = dnT_dt(*X)
    nP_k1 = dnP_dt(*X)

    X = (nI + (h / 2) * nI_k1, nT + (h / 2) * nT_k1, nP + (h / 2) * nP_k1)
    nI_k2 = dnI_dt(*X)
    nT_k2 = dnT_dt(*X)
    nP_k2 = dnP_dt(*X)

    X = (nI + (h / 2) * nI_k2, nT + (h / 2) * nT_k2, nP + (h / 2) * nP_k2)
    nI_k3 = dnI_dt(*X)
    nT_k3 = dnT_dt(*X)
    nP_k3 = dnP_dt(*X)

    X = (nI + h * nI_k3, nT + h * nT_k3, nP + h * nP_k3)
    nI_k4 = dnI_dt(*X)
    nT_k4 = dnT_dt(*X)
    nP_k4 = dnP_dt(*X)

    nI_new = nI + (h / 6) * (nI_k1 + 2 * nI_k2 + 2 * nI_k3 + nI_k4)
    nT_new = nT + (h / 6) * (nT_k1 + 2 * nT_k2 + 2 * nT_k3 + nT_k4)
    nP_new = nP + (h / 6) * (nP_k1 + 2 * nP_k2 + 2 * nP_k3 + nP_k4)

    nI = nI_new
    nT = nT_new
    nP = nP_new

In [ ]:
nI.sum() + nT.sum() + nP.sum()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plots = [
    (nI, "Idle Vehicles"),
    (nT.sum(axis=0), "Vehicles En Route to Pick-up"),
    (nP.sum(axis=0), "Vehicles En Route to Drop-off"),
    (nP.sum(axis=1), "Pickup Origin Intensity"),
]

for ax, (data, title) in zip(axes.flatten(), plots):
    gdf.plot(column=data, cmap="viridis", legend=True, ax=ax, edgecolor="none", vmin=min(0, data.min()))
    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
T_steps = len(nI_history)

def fields_at(step):
    nI = nI_history[step]
    nT = nT_history[step]
    nP = nP_history[step]

    return [nI, nT.sum(axis=0), nP.sum(axis=0), nP.sum(axis=1)]


titles = ["Idle Vehicles",
          "Vehicles En Route to Pick-up",
          "Vehicles En Route to Drop-off",
          "Pickup Origin Intensity"]

# fix color scales by computing vmin/vmax across all frames
maxs = np.full(4, -np.inf)
for t in range(T_steps):
    vals = fields_at(t)
    for k in range(4):
        v = np.asarray(vals[k])
        maxs[k] = max(maxs[k], np.nanmax(v))

cmap = plt.get_cmap("viridis")
norms = [mpl.colors.Normalize(vmin=0, vmax=maxs[k]) for k in range(4)]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

collections = []
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

# initial plots
init_vals = fields_at(0)
for k, ax in enumerate(axes):
    gdf.plot(column=init_vals[k], cmap=cmap, ax=ax, edgecolor="none", vmin=0, vmax=maxs[k], legend=False)
    collections.append(ax.collections[0])

# add fixed colorbars
cbs = []
for k, ax in enumerate(axes):
    sm = mpl.cm.ScalarMappable(norm=norms[k], cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
    cbs.append(cb)

supt = fig.suptitle(f"Fleet State Snapshot — t = {h * 0:.2f} hours", fontsize=18)
plt.tight_layout()

def update(frame):
    vals = fields_at(frame)

    for k in range(4):
        collections[k].set_array(np.asarray(vals[k], dtype=float))

    supt.set_text(f"Fleet State Snapshot — t = {h * history_interval * frame:.2f} hours")
    return collections + [supt]


anim = FuncAnimation(fig, update, frames=T_steps, interval=50, blit=False)
anim.save("fleet_state.mp4", writer="ffmpeg", fps=20)